# Gomoku AlphaZero Training (Kaggle Dual-T4 Optimized)

This notebook is an all-in-one, high-efficiency Alphazero implementation for a 15x15 Gomoku board. 

### 🌟 Key Enhancements
1. **Zero Deepcopy MCTS**: Uses inplace `do_move` and `undo_move` to speed up simulations significantly.
2. **ResNet-10 Architecture**: Deep residual network for superior pattern recognition.
3. **Dual T4 Acceleration**: Automatically uses `torch.nn.DataParallel` when Kaggle's dual GPUs are active.
4. **Batch Training**: Processes games with larger batch sizes (512+) to saturate GPU memory.
5. **Real-time Evaluation**: Periodically plays against a pure MCTS opponent to monitor win rates.

In [ ]:
import os
import time
import random
import numpy as np
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
from IPython import display

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 1. Hyperparameters

In [ ]:
@dataclass
class Config:
    board_size: int = 15
    n_in_row:   int = 5
    
    # Model architecture
    n_res_blocks: int = 10
    n_filters:    int = 256
    
    # MCTS settings
    c_puct:         float = 5.0
    n_playout:      int   = 400     # Simulations for self-play
    n_playout_eval: int   = 1000    # Simulations for evaluation/play
    pure_mcts_playout: int = 1000   # Opponent simulations
    
    # Exploration
    temperature:    float = 1.0
    temp_threshold: int   = 30
    noise_eps:      float = 0.25
    dirichlet_alpha: float = 0.3
    
    # Training params
    learn_rate:       float = 2e-3
    lr_multiplier:    float = 1.0
    train_batch_size: int   = 512
    update_epochs:    int   = 5
    kl_coeff:         float = 0.02
    buffer_size:      int   = 20000
    
    # Pipeline
    total_iterations: int = 5000
    games_per_iter:   int = 1
    check_freq:       int = 50
    n_eval_games:     int = 6
    
    checkpoint_dir: str = "./checkpoints"
    device: object = field(default=None, init=False)
    n_gpus: int    = field(default=0,    init=False)

    def __post_init__(self):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            self.n_gpus = torch.cuda.device_count()
        else:
            self.device = torch.device("cpu")
            self.n_gpus = 0
        os.makedirs(self.checkpoint_dir, exist_ok=True)

cfg = Config()

## 2. Environment (Gomoku Logic)

In [ ]:
class GomokuEnv:
    def __init__(self, board_size=15, n_in_row=5):
        self.board_size = board_size
        self.n_in_row = n_in_row
        self.players = [1, 2] # 1:Black, 2:White
        self.reset()
        
    def reset(self):
        self.board = np.zeros((self.board_size, self.board_size), dtype=np.int8)
        self.history = []
        self.current_player = self.players[0]
        self.availables = set(range(self.board_size * self.board_size))
        return self.current_state()
        
    def do_move(self, move):
        h, w = move // self.board_size, move % self.board_size
        self.board[h, w] = self.current_player
        self.availables.remove(move)
        self.history.append((self.current_player, move))
        self.current_player = 3 - self.current_player
        
    def undo_move(self):
        prev_player, move = self.history.pop()
        h, w = move // self.board_size, move % self.board_size
        self.board[h, w] = 0
        self.availables.add(move)
        self.current_player = prev_player
        
    def current_state(self):
        # 4 layers: current, opponent, last_move, turn_color
        state = np.zeros((4, self.board_size, self.board_size), dtype=np.float32)
        state[0] = (self.board == self.current_player).astype(np.float32)
        state[1] = (self.board == (3 - self.current_player)).astype(np.float32)
        if self.history:
            lm = self.history[-1][1]
            state[2, lm // self.board_size, lm % self.board_size] = 1.0
        if len(self.history) % 2 == 0:
            state[3, :, :] = 1.0
        return state
        
    def game_ended(self):
        if len(self.history) < self.n_in_row * 2 - 1: return False, -1
        last_p, last_m = self.history[-1]
        h, w = last_m // self.board_size, last_m % self.board_size
        for dh, dw in [(1,0), (0,1), (1,1), (1,-1)]:
            count = 1
            for i in range(1, self.n_in_row):
                r, c = h+i*dh, w+i*dw
                if 0<=r<self.board_size and 0<=c<self.board_size and self.board[r,c]==last_p: count+=1
                else: break
            for i in range(1, self.n_in_row):
                r, c = h-i*dh, w-i*dw
                if 0<=r<self.board_size and 0<=c<self.board_size and self.board[r,c]==last_p: count+=1
                else: break
            if count >= self.n_in_row: return True, last_p
        if not self.availables: return True, -1
        return False, -1

## 3. ResNet Policy-Value Model

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        res = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + res)

class PolicyValueNet(nn.Module):
    def __init__(self, board_size=15, n_res_blocks=10, n_filters=256):
        super().__init__()
        self.conv_in = nn.Conv2d(4, n_filters, 3, padding=1, bias=False)
        self.bn_in = nn.BatchNorm2d(n_filters)
        self.res = nn.Sequential(*[ResidualBlock(n_filters) for _ in range(n_res_blocks)])
        
        # Policy head
        self.p_conv = nn.Conv2d(n_filters, 2, 1, bias=False)
        self.p_bn = nn.BatchNorm2d(2)
        self.p_fc = nn.Linear(2 * board_size**2, board_size**2)
        
        # Value head
        self.v_conv = nn.Conv2d(n_filters, 1, 1, bias=False)
        self.v_bn = nn.BatchNorm2d(1)
        self.v_fc1 = nn.Linear(board_size**2, 256)
        self.v_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))
        x = self.res(x)
        p = self.p_fc(F.relu(self.p_bn(self.p_conv(x))).view(x.size(0), -1))
        v = torch.tanh(self.v_fc2(F.relu(self.v_fc1(F.relu(self.v_bn(self.v_conv(x))).view(x.size(0), -1)))))
        return F.log_softmax(p, dim=1), v

## 4. MCTS Player

In [ ]:
class TreeNode:
    def __init__(self, parent, prior_p):
        self.parent, self.children = parent, {}
        self.n_visits, self.Q, self.P = 0, 0, prior_p
    def expand(self, action_priors): 
        for a, p in action_priors: 
            if a not in self.children: self.children[a] = TreeNode(self, p)
    def select(self, c_puct):
        return max(self.children.items(), key=lambda x: x[1].Q + c_puct * x[1].P * np.sqrt(self.n_visits)/(1 + x[1].n_visits))
    def update_recursive(self, leaf_value):
        if self.parent: self.parent.update_recursive(-leaf_value)
        self.n_visits += 1
        self.Q += (leaf_value - self.Q) / self.n_visits

class MCTSPlayer:
    def __init__(self, policy_fn, c_puct=5, n_playout=400, is_selfplay=False):
        self.root, self.policy, self.c_puct, self.n_playout = TreeNode(None, 1.0), policy_fn, c_puct, n_playout
        self.is_selfplay = is_selfplay

    def playout(self, state):
        node, moves = self.root, []
        while node.children:
            a, node = node.select(self.c_puct)
            state.do_move(a); moves.append(a)
        end, winner = state.game_ended()
        if not end:
            p, v = self.policy(state)
            node.expand(p)
        else:
            v = 0.0 if winner == -1 else (1.0 if winner == state.current_player else -1.0)
        node.update_recursive(-v)
        for _ in reversed(moves): state.undo_move()

    def get_action(self, env, temp=1e-3, return_prob=False):
        for _ in range(self.n_playout): self.playout(env)
        acts, visits = zip(*[(a, n.n_visits) for a, n in self.root.children.items()])
        probs = F.softmax(torch.tensor(np.log(np.array(visits) + 1e-10) / temp), dim=0).numpy()
        if self.is_selfplay:
            probs = 0.75*probs + 0.25*np.random.dirichlet(0.3*np.ones(len(probs)))
        move = np.random.choice(acts, p=probs)
        self.root = self.root.children[move] if move in self.root.children else TreeNode(None, 1.0)
        self.root.parent = None
        if return_prob:
            pr = np.zeros(env.board_size**2); pr[list(acts)] = probs
            return move, pr
        return move

## 5. Training Logic

In [ ]:
def get_equi_data(data, size):
    ext = []
    for s, p, z in data:
        for i in [1,2,3,4]:
            s_rot = np.array([np.rot90(ch, i) for ch in s])
            p_rot = np.rot90(np.flipud(p.reshape(size, size)), i)
            ext.append((s_rot, np.flipud(p_rot).flatten(), z))
            s_flip = np.array([np.fliplr(ch) for ch in s_rot])
            p_flip = np.fliplr(p_rot)
            ext.append((s_flip, np.flipud(p_flip).flatten(), z))
    return ext

class TrainPipeline:
    def __init__(self, cfg):
        self.cfg = cfg
        self.buffer = deque(maxlen=cfg.buffer_size)
        self.net = PolicyValueNet(cfg.board_size, cfg.n_res_blocks, cfg.n_filters).to(cfg.device)
        if cfg.n_gpus > 1: self.net = nn.DataParallel(self.net)
        self.opt = optim.Adam(self.net.parameters(), weight_decay=1e-4, lr=cfg.learn_rate)
        self.metrics = defaultdict(list)
        
    def policy(self, board):
        self.net.eval()
        with torch.no_grad():
            p, v = self.net(torch.tensor(board.current_state(), dtype=torch.float32, device=self.cfg.device).unsqueeze(0))
        probs = np.exp(p.cpu().numpy().flatten())
        return [(m, probs[m]) for m in board.availables], v.item()

    def self_play(self):
        env = GomokuEnv(self.cfg.board_size, self.cfg.n_in_row)
        player = MCTSPlayer(self.policy, self.cfg.c_puct, self.cfg.n_playout, True)
        states, probs, players = [], [], []
        while True:
            t = self.cfg.temperature if len(states) < self.cfg.temp_threshold else 1e-3
            move, prob = player.get_action(env, t, True)
            states.append(env.current_state()); probs.append(prob); players.append(env.current_player)
            env.do_move(move)
            end, winner = env.game_ended()
            if end:
                z = np.zeros(len(players))
                if winner != -1:
                    z[np.array(players) == winner] = 1.0
                    z[np.array(players) != winner] = -1.0
                self.buffer.extend(get_equi_data(zip(states, probs, z), self.cfg.board_size))
                return len(states)

    def update(self):
        batch = random.sample(self.buffer, self.cfg.train_batch_size)
        s = torch.tensor(np.array([d[0] for d in batch]), dtype=torch.float32, device=self.cfg.device)
        p_target = torch.tensor(np.array([d[1] for d in batch]), dtype=torch.float32, device=self.cfg.device)
        z_target = torch.tensor(np.array([d[2] for d in batch]), dtype=torch.float32, device=self.cfg.device).unsqueeze(1)
        self.net.train()
        for _ in range(self.cfg.update_epochs):
            self.opt.zero_grad()
            p, v = self.net(s)
            loss = F.mse_loss(v, z_target) - torch.mean(torch.sum(p_target * p, 1))
            loss.backward(); self.opt.step()
        return loss.item()

    def evaluate(self):
        env = GomokuEnv(self.cfg.board_size, self.cfg.n_in_row)
        c_player = MCTSPlayer(self.policy, self.cfg.c_puct, self.cfg.n_playout_eval)
        # Pure MCTS player
        def pure_policy(b): return [(m, 1/len(b.availables)) for m in b.availables], 0
        p_player = MCTSPlayer(pure_policy, 5.0, self.cfg.pure_mcts_playout)
        win_cnt = 0
        for i in range(self.cfg.n_eval_games):
            env.reset()
            ps = {1: c_player, 2: p_player} if i % 2 == 0 else {1: p_player, 2: c_player}
            while True:
                env.do_move(ps[env.current_player].get_action(env))
                end, winner = env.game_ended()
                if end:
                    if winner != -1 and ps[winner] == c_player: win_cnt += 1
                    elif winner == -1: win_cnt += 0.5
                    break
        return win_cnt / self.cfg.n_eval_games

    def run(self):
        for i in range(self.cfg.total_iterations):
            l_ep = self.self_play()
            if len(self.buffer) > self.cfg.train_batch_size:
                loss = self.update()
                self.metrics['loss'].append(loss)
                if i % 10 == 0: 
                    print(f"Iter {i}: Loss {loss:.4f}, Buf {len(self.buffer)}")
            if (i+1) % self.cfg.check_freq == 0:
                wr = self.evaluate()
                self.metrics['win_rate'].append(wr)
                print(f"--- Eval --- Win Rate: {wr:.2f}")
                torch.save(self.net.state_dict(), f"model_{i+1}.pt")
                self.plot()

    def plot(self):
        display.clear_output(wait=True)
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1); plt.plot(self.metrics['loss']); plt.title('Loss')
        plt.subplot(1, 2, 2); plt.plot(self.metrics['win_rate']); plt.title('Win Rate')
        plt.show()

## 6. Execution

The training will automatically detect GPUs. You can interrupt with `KeyboardInterrupt` anytime.

In [ ]:
pipeline = TrainPipeline(cfg)
try:
    pipeline.run()
except KeyboardInterrupt:
    print("Halted.")